# Collective Behaviour — Correlation Length Analysis

This notebook runs the two-stage pipeline:

| Stage | Script | What it does |
|-------|--------|--------------|
| 1 | `extract_video_snippets.py` | Trims each raw video to its last N seconds and saves the clips in an auto-created subfolder |
| 2 | `tools.py :: run_pipeline` | Binarises frames, computes the 2-D structure factor, extracts the spatial correlation length ξ, and plots ξ vs the chosen control parameter |

**an in depth breakdown of purpose, functionality etc. is available in the README.md**!!



**Workflow assumptions**
- Raw `.mp4` files live in a single flat folder (one file per `(param_value, seed)` pair).
- Each filename encodes the control parameter and the seed, e.g.:
  ```
  run__strength_0.42__seed_3.mp4
  run__stimulus_strength_1.5e-3__seed_0.mp4
  ```
- Only the *last N seconds* of each clip contain the stationary collective state of interest.

## 0 · Imports

In [ ]:
import subprocess, os, sys
import numpy as np
import matplotlib.pyplot as plt
plt.ioff()   # suppress automatic inline display inside loops

from tools import run_pipeline

---
## Stage 1 · Extract last-N-seconds clips

**What this does**

Calls `extract_video_snippets.py` via the shell.  
The script scans `RAW_VIDEO_DIR` for `.mp4` files and writes trimmed clips into  
`<RAW_VIDEO_DIR>/last{N}sec/` (created automatically if absent).

**Parameters to set**

| Variable | Meaning |
|----------|---------|
| `RAW_VIDEO_DIR` | folder that contains your raw simulation recordings |
| `N_SECONDS` | how many seconds to keep from the end of each clip |

> **Tip — skip this cell if clips already exist.**  
> The output subfolder is preserved across runs; the script will simply
> overwrite files with the same name.

In [ ]:
# ── USER PARAMETERS ──────────────────────────────────────────────────────────
RAW_VIDEO_DIR = "./video_data"   # folder with raw .mp4 files
N_SECONDS     = 20               # seconds to keep from the end
# ─────────────────────────────────────────────────────────────────────────────

# Build the CLI command.
# output_folder is intentionally omitted → auto-derived as
#   <RAW_VIDEO_DIR>/last{N_SECONDS}sec/
cmd = [
    sys.executable, "extract_video_snippets.py",
    RAW_VIDEO_DIR,
    "--seconds", str(N_SECONDS),
]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr, file=sys.stderr)
else:
    print("✓ Extraction complete.")

# Derive the clip folder path so Stage 2 can pick it up automatically
CLIP_DIR = os.path.join(RAW_VIDEO_DIR, f"last{N_SECONDS}sec")
print(f"Clip directory: {CLIP_DIR}")

---
## Stage 2 · Correlation length vs control parameter

**What this does**

For every clip in `CLIP_DIR`:

1. **Binarise** each frame (Otsu or fixed threshold).
2. **Accumulate** the 2-D structure factor `S(k)` over all frames.
3. **Inverse-FFT** the mean `S(k)` → spatial autocorrelation `C(r)`.
4. **Read off** the 1/e radius of the radially averaged `C(r)` → ξ.

Clips are grouped by the value of `PARAM_PREFIX` extracted from the filename.  
Mean and standard deviation of ξ are computed over seeds and plotted.

**Parameters to set**

| Variable | Meaning | Example |
|----------|---------|-------|
| `CLIP_DIR` | folder with the trimmed clips | auto-set by Stage 1 |
| `PARAM_PREFIX` | token before the numeric value in the filename | `"strength"`, `"stimulus_strength"`, `"threshold"` |
| `SEEDS` | list of seed integers to include; `None` → all seeds | `[0,1,2,3,4]` |
| `PARAMS_TO_SKIP` | parameter values to exclude | `[100.0]` |
| `BINARIZATION` | `"otsu"` (recommended) or `"fixed"` | `"otsu"` |
| `CHECK_BINARIZATION` | show raw-vs-binary frame for the first clip per param value | `True` |
| `VISUALIZE_INTERMEDIATE` | show per-clip S(k), C(r), and decay plots | `False` |
| `DEBUG` | print per-seed ξ values | `False` |

In [ ]:
# ── USER PARAMETERS ──────────────────────────────────────────────────────────
# CLIP_DIR is carried over from Stage 1; override here if running standalone:
# CLIP_DIR = "./video_data/last20sec"

PARAM_PREFIX          = "strength"          # token that precedes the value in the filename
SEEDS                 = [0,1,2,3,4,5,6,7,8,9]  # seed integers to include; None → all
PARAMS_TO_SKIP        = [100.0]             # parameter values to exclude
BINARIZATION          = "otsu"              # "otsu" | "fixed"
CHECK_BINARIZATION    = True                # visual sanity-check per param value
VISUALIZE_INTERMEDIATE = False              # per-clip diagnostic plots
DEBUG                 = False               # print per-seed ξ
# ─────────────────────────────────────────────────────────────────────────────

sorted_params, xi_means, xi_by_param = run_pipeline(
    folder                 = CLIP_DIR,
    param_prefix           = PARAM_PREFIX,
    seeds                  = SEEDS,
    params_to_skip         = PARAMS_TO_SKIP,
    binarization           = BINARIZATION,
    check_binarization     = CHECK_BINARIZATION,
    visualize_intermediate = VISUALIZE_INTERMEDIATE,
    debug                  = DEBUG,
)

### Inspect results

In [ ]:
# Print a summary table
print(f"{'':>4}  {PARAM_PREFIX:>18}  {'mean ξ (px)':>12}  {'std ξ (px)':>11}")
print("-" * 52)
for i, p in enumerate(sorted_params):
    m = xi_by_param[p]['mean']
    s = xi_by_param[p]['std']
    marker = " ← max" if xi_means[i] == np.nanmax(xi_means) else ""
    print(f"{i:>4}  {p:>18.4g}  {m:>12.2f}  {s:>11.2f}{marker}")